In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

HERE = Path.cwd()


def find_module_dir(start=HERE):
    """Locate the folder holding definitions.py, wherever the notebook was opened from.

    Jupyter sets the working directory to wherever it was launched, which may be
    the repo root, analytics/, or this folder. Guessing one of those and failing
    on the other two is the first thing that goes wrong for someone else.
    """
    candidates = [start, start / "baseline_survey_scoring",
                  start / "analytics" / "baseline_survey_scoring"]
    candidates += [parent / "baseline_survey_scoring" for parent in start.parents]
    candidates += [parent / "analytics" / "baseline_survey_scoring" for parent in start.parents]
    for candidate in candidates:
        if (candidate / "definitions.py").exists():
            return candidate
    raise FileNotFoundError(
        f"definitions.py not found from {start}. Open this notebook from inside the "
        "REACT repository, or set MODULE_DIR by hand."
    )


MODULE_DIR = find_module_dir()
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

import definitions as D
import qualtrics as Q
import scoring as S

EXPORT_PATH = None        # Qualtrics export, .csv or .xlsx
HAND_SCORED_PATH = None   # hand-scored scores, .csv or .xlsx

OUTPUT_DIR = MODULE_DIR / "output"

# Qualtrics exports a QID header when the survey was not given variable names.
# Add "exact header text": "definitions.py column name" here when that happens
# and pass it to Q.load_export(..., overrides=COLUMN_OVERRIDES) in section 8;
# it is the only place a column mapping needs editing.
COLUMN_OVERRIDES = {}

print("definitions.py, qualtrics.py, scoring.py loaded")
print(f"  {len(D.ALL_SCORING_COLUMNS)} scoring columns, {len(D.ALL_PRESERVED_COLUMNS)} preserved")
print(f"  {len(Q.ITEM_TABLE)} items mapped across {Q.ITEM_TABLE['instrument'].nunique()} instruments")
print(f"  {len(S.SCORERS)} instrument scorers, {len(S.SCORE_RANGES)} documented score ranges")
print(f"  export      : {EXPORT_PATH or 'NOT SET'}")
print(f"  hand-scored : {HAND_SCORED_PATH or 'NOT SET'}")


definitions.py, qualtrics.py, scoring.py loaded
  205 scoring columns, 11 preserved
  208 items mapped across 24 instruments
  23 instrument scorers, 46 documented score ranges
  export      : NOT SET
  hand-scored : NOT SET


In [2]:
# Sections 1-6 — response option tables, the Qualtrics item/text mapping and
# loader, and the generic scoring engine + all 23 instrument scorers now live
# in qualtrics.py (as Q) and scoring.py (as S), imported in section 0. This
# cell just confirms they loaded; nothing here is redefined.

_option_tables = sum(
    1 for _name, _value in vars(Q).items()
    if _name.endswith("_OPTIONS") and isinstance(_value, dict)
)
print(f"qualtrics.py: {len(Q.ITEM_TABLE)} items, {_option_tables} response option tables")
print(f"scoring.py:   {len(S.SCORERS)} instrument scorers, {len(S.ALERT_RULES)} alert rules")


qualtrics.py: 208 items, 32 response option tables
scoring.py:   23 instrument scorers, 4 alert rules


In [3]:
# Section 7 — self-test.


CHECKS = {"pass": 0, "fail": 0}


def check(label, actual, expected, tolerance=1e-9):
    """Numbers compare within a tolerance, everything else by equality."""
    numeric = isinstance(expected, (int, float, np.number)) and not isinstance(expected, bool)
    if not numeric:
        ok = actual == expected
    elif pd.isna(expected):
        ok = pd.isna(actual)
    else:
        ok = (not pd.isna(actual)) and abs(float(actual) - float(expected)) <= tolerance
    CHECKS["pass" if ok else "fail"] += 1
    print(f"  {'PASS' if ok else 'FAIL'}  {label}: got {actual!r}, expected {expected!r}")
    return ok


# Valid integer range per item, taken from the option tables so a change there
# cannot drift away from what the range checks below assume.
COLUMN_SCALE = {}
for _row in Q.ITEM_TABLE.itertuples():
    if _row.column not in set(D.ALL_SCORING_COLUMNS):
        continue
    if _row.options:
        values = list(_row.options.values())
        COLUMN_SCALE[_row.column] = (min(values), max(values))
for _column in D.SSIS:
    COLUMN_SCALE[_column] = S.SSIS_RANGE
for _column in D.MACARTHUR:
    COLUMN_SCALE[_column] = S.LADDER_RANGE
COLUMN_SCALE[D.TFEQ_RESTRAINT_1_TO_8] = (1, 8)


def fixture(rows=1, **assignments):
    """A frame of all-NA items, with the named columns or groups filled in."""
    frame = pd.DataFrame({c: [np.nan] * rows
                          for c in list(D.ALL_SCORING_COLUMNS) + [D.GENDER, D.PARTICIPANT_ID]})
    frame[D.PARTICIPANT_ID] = [f"p{i}" for i in range(rows)]
    for key, value in assignments.items():
        columns = getattr(D, key.upper(), key)
        for column in (columns if isinstance(columns, (list, tuple)) else [columns]):
            frame[column] = value
    return frame


print("Reverse coding")
check("reverse low end of 1-4", S.reverse(pd.Series([1.0]), 1, 4).iloc[0], 4)
check("reverse high end of 1-4", S.reverse(pd.Series([4.0]), 1, 4).iloc[0], 1)
check("reverse midpoint of 1-4 is fixed", S.reverse(pd.Series([2.5]), 1, 4).iloc[0], 2.5)
check("reverse low end of 0-4", S.reverse(pd.Series([0.0]), 0, 4).iloc[0], 4)
check("reverse low end of 1-7", S.reverse(pd.Series([1.0]), 1, 7).iloc[0], 7)

print("\nMissing data: one blank item nulls its subscale and nothing else")
_missing = fixture(ders=3)
_missing.loc[0, D.DERS_CLARITY[0]] = np.nan
_scored_missing = S.score_ders(_missing)
check("blank clarity item nulls ders_clarity_sum", _scored_missing["ders_clarity_sum"].iloc[0], np.nan)
check("sibling ders_goals_sum survives", _scored_missing["ders_goals_sum"].iloc[0], 9)
check("ders_total is nulled too", _scored_missing["ders_total"].iloc[0], np.nan)

print("\nSUPPS-P direction (1 = strongly agree, 4 = strongly disagree)")
_supps = fixture(supps=1)
_supps_scored = S.score_supps(_supps)
check("negative urgency is reversed, so agreeing scores 4",
      _supps_scored["supps_negative_urgency_mean"].iloc[0], 4)
check("lack of perseverance is not reversed, so agreeing scores 1",
      _supps_scored["supps_lack_perseverance_mean"].iloc[0], 1)
check("sensation seeking is reversed",
      _supps_scored["supps_sensation_seeking_mean"].iloc[0], 4)
check("lack of premeditation is not reversed",
      _supps_scored["supps_lack_premeditation_mean"].iloc[0], 1)

print("\nPSS-10 reversal on a 0-4 scale")
check("all zeros gives 16, not 0, because four items reverse to 4",
      S.score_pss(fixture(pss=0))["pss_total"].iloc[0], 16)
check("all fours gives 24", S.score_pss(fixture(pss=4))["pss_total"].iloc[0], 24)

print("\nBAQ even-tempered item reverses on 1-7")
check("all ones gives 12 + 6 for the reversed item",
      S.score_baq(fixture(baq=1))["baq_total"].iloc[0], 11 * 1 + 7)

print("\nPROMIS conversion table endpoints and a middle row")
_promis_low = fixture(promis_sleep=1)
_promis_low[D.PROMIS_SLEEP[0]] = 5
_promis_low[D.PROMIS_SLEEP[1]] = 5
_low = S.score_promis(_promis_low)
check("best sleep gives raw 4", _low["promis_sleep_raw"].iloc[0], 4)
check("raw 4 converts to T 32.0", _low["promis_sleep_t"].iloc[0], 32.0)
check("raw 4 carries SE 5.2", _low["promis_sleep_se"].iloc[0], 5.2)

_promis_high = fixture(promis_sleep=5)
_promis_high[D.PROMIS_SLEEP[0]] = 1
_promis_high[D.PROMIS_SLEEP[1]] = 1
_high = S.score_promis(_promis_high)
check("worst sleep gives raw 20", _high["promis_sleep_raw"].iloc[0], 20)
check("raw 20 converts to T 73.3", _high["promis_sleep_t"].iloc[0], 73.3)
check("raw 12 converts to T 54.3", S.score_promis(fixture(promis_sleep=3))["promis_sleep_t"].iloc[0], 54.3)

print("\nTFEQ 1-to-8 ladder recode at every boundary")
for _raw, _expected in [(1, 1), (2, 1), (3, 2), (4, 2), (5, 3), (6, 3), (7, 4), (8, 4)]:
    check(f"ladder {_raw} recodes to {_expected}",
          S.recode_tfeq_ladder(pd.Series([float(_raw)])).iloc[0], _expected)

print("\nTFEQ 0-100 transform at both ends")
_tfeq_min = fixture(tfeq=1)
_tfeq_max = fixture(tfeq=4)
_tfeq_max[D.TFEQ_RESTRAINT_1_TO_8] = 8
check("cognitive restraint floor is 0",
      S.score_tfeq(_tfeq_min)["tfeq_cognitive_restraint_0_100"].iloc[0], 0)
check("cognitive restraint ceiling is 100",
      S.score_tfeq(_tfeq_max)["tfeq_cognitive_restraint_0_100"].iloc[0], 100)
check("uncontrolled eating ceiling is 100",
      S.score_tfeq(_tfeq_max)["tfeq_uncontrolled_eating_0_100"].iloc[0], 100)
check("emotional eating floor is 0",
      S.score_tfeq(_tfeq_min)["tfeq_emotional_eating_0_100"].iloc[0], 0)

print("\nASRS item-specific cuts")
_asrs = fixture(asrs=2)          # Sometimes on all six
_asrs_scored = S.score_asrs(_asrs)
check("Sometimes counts for items 1-3 only, so the count is 3",
      _asrs_scored["asrs_count"].iloc[0], 3)
check("a count of 3 is not a positive screen", _asrs_scored["asrs_positive"].iloc[0], False)
_asrs_mixed = fixture(asrs=2)
for _column in D.ASRS[3:]:
    _asrs_mixed[_column] = 3      # Often on items 4-6
_mixed_scored = S.score_asrs(_asrs_mixed)
check("Often on items 4-6 brings the count to 6", _mixed_scored["asrs_count"].iloc[0], 6)
check("a count of 6 is a positive screen", _mixed_scored["asrs_positive"].iloc[0], True)

print("\nAUDIT-C gender-conditional cut, both branches at exactly 3")
_audit = fixture(rows=3, audit_c=1)
_audit[D.GENDER] = ["Man", "Woman", None]
_audit_scored = S.score_audit_c(_audit)
check("total is 3 for all three", _audit_scored["audit_c_total"].iloc[0], 3)
check("Man at 3 does not alert (cut is 4)", _audit_scored["audit_c_alert"].iloc[0], False)
check("Woman at 3 alerts (cut is 3)", _audit_scored["audit_c_alert"].iloc[1], True)
check("unspecified gender at 3 alerts, the deliberate sensitive cut",
      _audit_scored["audit_c_alert"].iloc[2], True)

print("\nThreshold screens")
_scoff = fixture(scoff=0)
_scoff.loc[0, D.SCOFF[0]] = 1
check("one SCOFF yes is not a positive screen", S.score_scoff(_scoff)["scoff_positive"].iloc[0], False)
_scoff.loc[0, D.SCOFF[1]] = 1
check("two SCOFF yeses are a positive screen", S.score_scoff(_scoff)["scoff_positive"].iloc[0], True)

_hunger = fixture(hunger_vital_sign=0)
check("both items Never true is not a positive screen",
      S.score_hunger(_hunger)["hunger_positive"].iloc[0], False)
_hunger.loc[0, D.HUNGER_VITAL_SIGN[0]] = 1
check("one item Sometimes true is a positive screen",
      S.score_hunger(_hunger)["hunger_positive"].iloc[0], True)

print("\nPGSI bands")
for _total, _band in [(0, "non-problem"), (1, "low risk"), (4, "low risk"),
                      (5, "moderate risk"), (7, "moderate risk"), (8, "problem gambling"),
                      (27, "problem gambling")]:
    check(f"a PGSI total of {_total} is {_band}", S.pgsi_band(_total), _band)

print("\nrMEQ endpoints, including the item scored 6/4/2/0")
_rmeq_min = fixture(rmeq=np.nan)
for _column, _options in zip(D.RMEQ, Q.ITEM_OPTIONS["rmeq"]):
    _rmeq_min[_column] = min(_options.values())
_rmeq_max = fixture(rmeq=np.nan)
for _column, _options in zip(D.RMEQ, Q.ITEM_OPTIONS["rmeq"]):
    _rmeq_max[_column] = max(_options.values())
check("rMEQ floor is 4", S.score_rmeq(_rmeq_min)["rmeq_total"].iloc[0], 4)
check("rMEQ ceiling is 25", S.score_rmeq(_rmeq_max)["rmeq_total"].iloc[0], 25)

print("\nSSIS, emitted both ways because the codebook does not say which")
check("all eights sums to 56", S.score_ssis(fixture(ssis=8))["ssis_sum"].iloc[0], 56)
check("all eights means 8", S.score_ssis(fixture(ssis=8))["ssis_mean"].iloc[0], 8)

print("\nEvery score lands inside its documented range, over 300 random respondents")
_rng = np.random.default_rng(0)
_random = pd.DataFrame({
    column: _rng.integers(lo, hi + 1, size=300).astype(float)
    for column, (lo, hi) in COLUMN_SCALE.items()
})
_random[D.PARTICIPANT_ID] = [f"r{i}" for i in range(len(_random))]
_random[D.GENDER] = _rng.choice(["Man", "Woman", "Nonbinary", None], size=300)
_random_scores, _random_alerts = S.score_participants(_random)
_out_of_range = []
for _name, (_lo, _hi) in S.SCORE_RANGES.items():
    _values = _random_scores[_name].dropna()
    if len(_values) and not _values.between(_lo - 1e-9, _hi + 1e-9).all():
        _out_of_range.append((_name, float(_values.min()), float(_values.max()), _lo, _hi))
check("no score escapes its range", _out_of_range, [])
check("every random respondent produced a PHQ-9 total",
      int(_random_scores["phq9_total"].notna().sum()), 300)

print("\nText responses round-trip through the loader")
_text = pd.DataFrame({
    "In the last month, how often have you felt nervous and \"stressed\"?": ["Very often"],
    "I am an even-tempered person.": ["Extremely characteristic of me"],
    "Do you make yourself Sick because you feel uncomfortably full?": ["Yes"],
    "One hears about “morning” and “evening” types of people. Which one of these "
    "types do you consider yourself to be?": ["Definitely a morning type"],
})
_resolved, _unmatched = Q.resolve_columns(list(_text.columns))
check("all four text headers resolved", len(_unmatched), 0)
_renamed = _text.rename(columns=_resolved)
_options_by_column = dict(zip(Q.ITEM_TABLE["column"], Q.ITEM_TABLE["options"]))
check("Very often on PSS is 4",
      Q.to_numeric(_renamed["pss_3"], _options_by_column["pss_3"])[0].iloc[0], 4)
check("Extremely characteristic on BAQ is 7",
      Q.to_numeric(_renamed["baq_anger_1"], _options_by_column["baq_anger_1"])[0].iloc[0], 7)
check("Yes on SCOFF is 1",
      Q.to_numeric(_renamed["scoff_1"], _options_by_column["scoff_1"])[0].iloc[0], 1)
check("Definitely a morning type on rMEQ is 6",
      Q.to_numeric(_renamed["rmeq_5"], _options_by_column["rmeq_5"])[0].iloc[0], 6)

print(f"\n{CHECKS['pass']} passed, {CHECKS['fail']} failed")
if CHECKS["fail"]:
    print("!! Fix these before trusting anything below.")


Reverse coding
  PASS  reverse low end of 1-4: got np.float64(4.0), expected 4
  PASS  reverse high end of 1-4: got np.float64(1.0), expected 1
  PASS  reverse midpoint of 1-4 is fixed: got np.float64(2.5), expected 2.5
  PASS  reverse low end of 0-4: got np.float64(4.0), expected 4
  PASS  reverse low end of 1-7: got np.float64(7.0), expected 7

Missing data: one blank item nulls its subscale and nothing else
  PASS  blank clarity item nulls ders_clarity_sum: got np.float64(nan), expected nan
  PASS  sibling ders_goals_sum survives: got np.int64(9), expected 9
  PASS  ders_total is nulled too: got np.float64(nan), expected nan

SUPPS-P direction (1 = strongly agree, 4 = strongly disagree)
  PASS  negative urgency is reversed, so agreeing scores 4: got np.float64(4.0), expected 4
  PASS  lack of perseverance is not reversed, so agreeing scores 1: got np.float64(1.0), expected 1
  PASS  sensation seeking is reversed: got np.float64(4.0), expected 4
  PASS  lack of premeditation is not r

In [4]:
# Section 8 — score the export.

scores = None
alerts = None

if EXPORT_PATH is None:
    print("EXPORT_PATH is not set, so nothing was scored.")
    print("Put the Qualtrics export somewhere readable, set EXPORT_PATH in section 0,")
    print("and re-run from there.")
elif not Path(EXPORT_PATH).exists():
    print(f"EXPORT_PATH does not exist: {EXPORT_PATH}")
else:
    responses = Q.load_export(EXPORT_PATH, overrides=COLUMN_OVERRIDES)
    scores, alerts = S.score_participants(responses)
    print(f"\nscored {len(scores)} participants, {len(scores.columns)} output columns")
    print(f"{len(alerts)} alert(s) raised")
    display(scores.head(10))
    if len(alerts):
        display(alerts)


EXPORT_PATH is not set, so nothing was scored.
Put the Qualtrics export somewhere readable, set EXPORT_PATH in section 0,
and re-run from there.


In [5]:
# Section 9 — validate against Gabby's hand scores.


VALIDATION_TOLERANCE = 1e-6


def compare_to_hand_scores(computed, hand, tolerance=VALIDATION_TOLERANCE):
    """Cell-by-cell comparison on the participant id, over the shared columns."""
    key = D.PARTICIPANT_ID
    if key not in hand.columns:
        raise ValueError(f"the hand-scored file has no {key} column; columns are {list(hand.columns)}")

    left = computed.set_index(key)
    right = hand.set_index(key)

    shared_ids = left.index.intersection(right.index)
    shared_columns = [c for c in left.columns if c in right.columns]
    print(f"participants: {len(left)} computed, {len(right)} hand-scored, {len(shared_ids)} shared")
    print(f"score columns: {len(shared_columns)} shared")
    for label, missing in [("only computed", left.index.difference(right.index)),
                           ("only hand-scored", right.index.difference(left.index))]:
        if len(missing):
            print(f"  !! {label}: {sorted(missing)}")
    for label, missing in [("computed but not hand-scored", set(left.columns) - set(right.columns)),
                           ("hand-scored but not computed", set(right.columns) - set(left.columns))]:
        if missing:
            print(f"  -- {label}: {sorted(missing)}")

    mismatches = []
    for participant in shared_ids:
        for column in shared_columns:
            ours, theirs = left.loc[participant, column], right.loc[participant, column]
            if pd.isna(ours) and pd.isna(theirs):
                continue
            if pd.isna(ours) or pd.isna(theirs):
                mismatches.append((participant, column, ours, theirs))
                continue
            try:
                same = abs(float(ours) - float(theirs)) <= tolerance
            except (TypeError, ValueError):
                same = str(ours).strip().lower() == str(theirs).strip().lower()
            if not same:
                mismatches.append((participant, column, ours, theirs))

    compared = len(shared_ids) * len(shared_columns)
    print(f"\ncompared {compared} cells, {len(mismatches)} mismatch(es)")
    if mismatches:
        for participant, column, ours, theirs in mismatches[:60]:
            print(f"  {participant}  {column}: script {ours!r} vs hand {theirs!r}")
        if len(mismatches) > 60:
            print(f"  ... and {len(mismatches) - 60} more")
    return pd.DataFrame(mismatches, columns=[key, "score", "script", "hand_scored"])


validation = None

if HAND_SCORED_PATH is None or scores is None:
    print("NOT VALIDATED.")
    if scores is None:
        print("  No scores were produced; set EXPORT_PATH in section 0.")
    if HAND_SCORED_PATH is None:
        print("  HAND_SCORED_PATH is not set, so there is nothing to compare against.")
    print("  Until both are set and this reports zero mismatches, nothing in this")
    print("  notebook has been checked against Gabby and none of it is deployable.")
elif not Path(HAND_SCORED_PATH).exists():
    print(f"NOT VALIDATED: HAND_SCORED_PATH does not exist: {HAND_SCORED_PATH}")
else:
    hand_path = Path(HAND_SCORED_PATH)
    hand_scores = (pd.read_excel(hand_path) if hand_path.suffix.lower() in {".xlsx", ".xls"}
                   else pd.read_csv(hand_path))
    validation = compare_to_hand_scores(scores, hand_scores)
    if validation.empty:
        print("\nEXACT MATCH against the hand scores.")
    else:
        print("\nNOT VALIDATED: the differences above have to be explained before deployment.")
        display(validation)

NOT VALIDATED.
  No scores were produced; set EXPORT_PATH in section 0.
  HAND_SCORED_PATH is not set, so there is nothing to compare against.
  Until both are set and this reports zero mismatches, nothing in this
  notebook has been checked against Gabby and none of it is deployable.


In [6]:
# =============================================================================
# Section 10 — write the output files.
# =============================================================================

if scores is None:
    print("Nothing to write: no export was scored.")
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    scores_path = OUTPUT_DIR / "scores.csv"
    alerts_path = OUTPUT_DIR / "alerts.csv"
    scores.to_csv(scores_path, index=False)
    alerts.to_csv(alerts_path, index=False)
    print(f"wrote {scores_path}  ({len(scores)} rows, {len(scores.columns)} columns)")
    print(f"wrote {alerts_path}  ({len(alerts)} rows)")
    if validation is not None and validation.empty:
        print("Validated against the hand scores.")
    else:
        print("NOT validated against the hand scores; treat these files as a dry run.")

Nothing to write: no export was scored.


In [7]:
# =============================================================================
# Section 11 — what still needs a person.
# =============================================================================

OPEN_ITEMS = [
    ("SUPPS-P reverse direction",
     "The docx and definitions.py reverse Negative Urgency, Sensation Seeking and Positive "
     "Urgency (12 items). The superseded xlsx reverses Lack of Perseverance and Lack of "
     "Premeditation instead (8 items). Implemented the docx reading. Four subscale directions "
     "ride on it, so Prof. Chang or Gabby should confirm."),
    ("PHQ-9 scoring line",
     "The docx says the total is 'the sum of all four items'. Its own item list has nine, and "
     "the older xlsx carried PHQ-4 at this position. Scored as nine. The sentence is stale."),
    ("DMQ-R SF response scale",
     "The docx gives five points, the xlsx gives three. Implemented five, matching Kuntsche and "
     "Kuntsche (2009). No ruling needed, noted for the record."),
    ("SSIS sum or mean",
     "The codebook fixes the 1-to-8 scale but never says which. Both are emitted; the comparison "
     "against Gabby will show which one she computed."),
    ("PHQ-9 item 9",
     "Asks about self-harm. The checklist names four alerts and does not include it, so no safety "
     "alert was invented here. Worth asking Prof. Chang whether the protocol expects one."),
    ("Qualtrics column headers",
     "No export exists yet, so column matching is by item text. If the export ships QID headers, "
     "add them to COLUMN_OVERRIDES in section 0. The loader names every column it cannot place."),
]

for _title, _detail in OPEN_ITEMS:
    print(f"- {_title}\n    {_detail}\n")

- SUPPS-P reverse direction
    The docx and definitions.py reverse Negative Urgency, Sensation Seeking and Positive Urgency (12 items). The superseded xlsx reverses Lack of Perseverance and Lack of Premeditation instead (8 items). Implemented the docx reading. Four subscale directions ride on it, so Prof. Chang or Gabby should confirm.

- PHQ-9 scoring line
    The docx says the total is 'the sum of all four items'. Its own item list has nine, and the older xlsx carried PHQ-4 at this position. Scored as nine. The sentence is stale.

- DMQ-R SF response scale
    The docx gives five points, the xlsx gives three. Implemented five, matching Kuntsche and Kuntsche (2009). No ruling needed, noted for the record.

- SSIS sum or mean
    The codebook fixes the 1-to-8 scale but never says which. Both are emitted; the comparison against Gabby will show which one she computed.

- PHQ-9 item 9
    Asks about self-harm. The checklist names four alerts and does not include it, so no safety alert wa